# RAG Chạy Hoàn Toàn Local — Không Cần API -> đổi
## Nhưng model là 3 tỷ tham số -> Không thông minh như gemini được

Notebook này xây dựng hệ thống RAG **chạy 100% trên máy Colab**, không cần API key hay tài khoản gì thêm.

| Thành phần | Model | Kích thước | Mô tả |
|---|---|---|---|
| **LLM** | `Qwen2.5-3B-Instruct` | ~6 GB | Trả lời câu hỏi |
| **Embedding** | `all-MiniLM-L6-v2` | ~90 MB | Chuyển văn bản → vector |

**Tính năng:**
- ✅ Đọc nhiều loại file: PDF, Word (.docx), Excel (.xlsx), Ảnh (.jpg/.png)
- ✅ Chatbot có memory (nhớ lịch sử hội thoại)
- ✅ Chạy được trên T4 GPU miễn phí của Google Colab

> ⚠️ **Bước đầu tiên:** Vào `Runtime → Change runtime type → T4 GPU` trước khi chạy!

---
## BƯỚC 1 — Cài Thư Viện

In [ ]:
!pip install transformers accelerate sentence-transformers \
             pypdf python-docx openpyxl Pillow \
             bitsandbytes --quiet

print("✅ Cài đặt xong!")

---
## BƯỚC 2 — Tải Model Về Máy

Bước này tải 2 model từ HuggingFace về Colab (chỉ cần làm 1 lần):
- **Embedding model** (~90 MB) — tải nhanh
- **LLM Qwen2.5-3B** (~6 GB) — mất khoảng 3–5 phút

> 💡 Không cần đăng nhập hay token gì cả — cả hai model đều public hoàn toàn.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# Kiểm tra GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚙️  Thiết bị: {device.upper()}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   ⚠️ Không có GPU! Hãy vào Runtime → Change runtime type → T4 GPU")

# ── Tải Embedding Model (nhỏ, nhanh) ──────────────────────────────
print("\n📦 Đang tải Embedding model (~90 MB)...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
print("✅ Embedding model sẵn sàng!")

# ── Tải LLM (lớn hơn, mất vài phút) ──────────────────────────────
LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"
print(f"\n📦 Đang tải LLM: {LLM_NAME} (~6 GB, chờ 3-5 phút)...")

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=torch.float16,   # Dùng float16 để tiết kiệm VRAM
    device_map="auto",            # Tự động phân bổ GPU/CPU
)
llm.eval()  # Chuyển sang chế độ inference (không train)

print(f"✅ LLM sẵn sàng!")
if device == "cuda":
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   VRAM đang dùng: {used:.1f} / {total:.1f} GB")

---
## BƯỚC 3 — Upload và Đọc File

Upload file lên Colab bằng cách kéo thả vào panel 📁 bên trái.

Hỗ trợ: **PDF**, **Word (.docx)**, **Excel (.xlsx)**

In [ ]:
import os
from pypdf import PdfReader
import docx
import openpyxl
from PIL import Image


def doc_file_pdf(path):
    reader = PdfReader(path)
    return "\n".join(p.extract_text() for p in reader.pages if p.extract_text())


def doc_file_word(path):
    doc = docx.Document(path)
    return "\n".join(p.text for p in doc.paragraphs if p.text.strip())


def doc_file_excel(path):
    wb = openpyxl.load_workbook(path, data_only=True)
    text = ""
    for sheet in wb.sheetnames:
        text += f"=== Sheet: {sheet} ===\n"
        for row in wb[sheet].iter_rows(values_only=True):
            vals = [str(v) for v in row if v is not None]
            if vals:
                text += " | ".join(vals) + "\n"
    return text


def doc_file(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        print(f"  📄 PDF: {path}")
        return doc_file_pdf(path)
    elif ext == ".docx":
        print(f"  📝 Word: {path}")
        return doc_file_word(path)
    elif ext in [".xlsx", ".xls"]:
        print(f"  📊 Excel: {path}")
        return doc_file_excel(path)
    else:
        raise ValueError(f"Loại file chưa hỗ trợ: {ext}")


# ================================================================
# 👇 THAY BẰNG ĐƯỜNG DẪN FILE CỦA BẠN
# ================================================================
DANH_SACH_FILE = [
    "/content/Quang Hải.pdf",
    # "/content/file.docx",
    # "/content/bang.xlsx",
]

toan_bo_van_ban = ""
for f in DANH_SACH_FILE:
    if not os.path.exists(f):
        print(f"  ⚠️  Không tìm thấy: {f}")
        continue
    van_ban = doc_file(f)
    toan_bo_van_ban += van_ban + "\n"
    print(f"     → {len(van_ban):,} ký tự")

print(f"\n📚 Tổng: {len(toan_bo_van_ban):,} ký tự từ {len(DANH_SACH_FILE)} file")

---
## BƯỚC 4 — Cắt Đoạn & Tạo Embeddings

Văn bản được cắt thành các đoạn nhỏ ~500 ký tự, sau đó chuyển thành vector số để tìm kiếm.

In [ ]:
import numpy as np

# ── Cắt đoạn ─────────────────────────────────────────────────────
def cat_doan(van_ban, chunk_size=500, overlap=80):
    doan_list, i = [], 0
    while i < len(van_ban):
        doan = van_ban[i : i + chunk_size]
        if doan.strip():
            doan_list.append(doan)
        i += chunk_size - overlap
    return doan_list

cac_doan = cat_doan(toan_bo_van_ban)
print(f"✂️  Đã cắt thành {len(cac_doan)} đoạn")

# ── Tạo Embeddings ───────────────────────────────────────────────
# SentenceTransformer xử lý cả batch cùng lúc → nhanh hơn gọi từng cái
print("\n🔢 Đang tạo embeddings...")
tat_ca_embeddings = embedding_model.encode(
    cac_doan,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)
print(f"✅ Xong! Ma trận embeddings: {tat_ca_embeddings.shape}")
print(f"   ({tat_ca_embeddings.shape[0]} đoạn × {tat_ca_embeddings.shape[1]} chiều)")

---
## BƯỚC 5 — Chatbot RAG Có Memory

Chatbot này:
- **Tìm** các đoạn liên quan nhất trong tài liệu
- **Nhớ** lịch sử hội thoại (tối đa 5 lượt gần nhất)
- **Trả lời** dựa trên ngữ cảnh tài liệu + lịch sử

Gõ **`xoa`** để reset lịch sử | Gõ **`thoat`** để dừng

In [ ]:
# ── Hàm tìm kiếm ─────────────────────────────────────────────────
def tim_doan_lien_quan(cau_hoi, top_k=3):
    """Tìm top_k đoạn liên quan nhất dùng cosine similarity."""
    vec_cau_hoi = embedding_model.encode([cau_hoi], convert_to_numpy=True)[0]

    # Tính cosine similarity với tất cả đoạn
    # (dùng phép nhân ma trận cho nhanh)
    norm_cau_hoi = vec_cau_hoi / np.linalg.norm(vec_cau_hoi)
    norm_doan    = tat_ca_embeddings / np.linalg.norm(tat_ca_embeddings, axis=1, keepdims=True)
    diem = norm_doan @ norm_cau_hoi

    top_idx = np.argsort(diem)[::-1][:top_k]
    return [cac_doan[i] for i in top_idx]


# ── Hàm gọi LLM ──────────────────────────────────────────────────
def goi_llm(prompt, max_new_tokens=400):
    """Gửi prompt cho LLM, nhận về chuỗi trả lời."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,         # Greedy decode — nhanh và ổn định
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Chỉ lấy phần LLM sinh ra (bỏ phần prompt đầu vào)
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


# ── Hàm RAG có Memory ─────────────────────────────────────────────
SO_LUOT_NHO = 5  # Nhớ tối đa 5 lượt

def tra_loi_rag(cau_hoi, lich_su):
    """
    Xử lý 1 lượt hỏi đáp:
    1. Tìm ngữ cảnh liên quan
    2. Ghép lịch sử + ngữ cảnh + câu hỏi → prompt
    3. Gọi LLM → câu trả lời
    4. Cập nhật lịch sử
    """
    # Bước 1: tìm ngữ cảnh
    doan_lq = tim_doan_lien_quan(cau_hoi)
    ngu_canh = "\n---\n".join(doan_lq)

    # Bước 2: ghép lịch sử
    lich_su_gan = lich_su[-(SO_LUOT_NHO * 2):]
    lich_su_text = ""
    for luot in lich_su_gan:
        prefix = "Người dùng" if luot["vai_tro"] == "nguoi_dung" else "Trợ lý"
        lich_su_text += f"{prefix}: {luot['noi_dung']}\n"

    # Bước 3: xây dựng prompt theo định dạng Qwen2.5
    system_msg = (
        "Bạn là trợ lý AI trả lời dựa trên tài liệu được cung cấp. "
        "Chỉ trả lời dựa trên NGỮ CẢNH. "
        "Nếu không có thông tin, nói: 'Tôi không tìm thấy thông tin này trong tài liệu.' "
        "Trả lời ngắn gọn bằng tiếng Việt."
    )

    user_msg = f"""NGỮ CẢNH TÀI LIỆU:
{ngu_canh}
"""
    if lich_su_text:
        user_msg += f"""\nLỊCH SỬ HỘI THOẠI:
{lich_su_text}"""

    user_msg += f"\nCÂU HỎI: {cau_hoi}"

    # Dùng chat template của Qwen
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": user_msg},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Bước 4: gọi LLM
    cau_tra_loi = goi_llm(prompt)

    # Bước 5: cập nhật lịch sử
    lich_su.append({"vai_tro": "nguoi_dung", "noi_dung": cau_hoi})
    lich_su.append({"vai_tro": "bot",        "noi_dung": cau_tra_loi})

    return cau_tra_loi, lich_su


# ── Chạy Chatbot ──────────────────────────────────────────────────
lich_su_hoi_thoai = []

print("🤖 CHATBOT RAG LOCAL")
print(f"   Model: Qwen2.5-3B-Instruct | Số đoạn: {len(cac_doan)}")
print(f"   Memory: {SO_LUOT_NHO} lượt | 'xoa' = reset | 'thoat' = dừng")
print("=" * 55)

while True:
    print()
    cau_hoi = input("❓ Bạn: ").strip()

    if cau_hoi.lower() in ["thoat", "exit", "quit", "q"]:
        print("👋 Tạm biệt!")
        break

    if cau_hoi.lower() == "xoa":
        lich_su_hoi_thoai = []
        print("🗑️  Đã xóa lịch sử.")
        continue

    if not cau_hoi:
        continue

    so_luot = len(lich_su_hoi_thoai) // 2
    if so_luot > 0:
        print(f"   (🧠 nhớ {so_luot} lượt trước)")

    print("   ⏳ Đang suy nghĩ...")
    try:
        tra_loi, lich_su_hoi_thoai = tra_loi_rag(cau_hoi, lich_su_hoi_thoai)
        print(f"\n💬 Bot: {tra_loi}")
    except Exception as e:
        print(f"❌ Lỗi: {e}")
    print("-" * 55)